# Module 5: Model Management and Deployment

Once we have fine-tuned our LLMs (either via LoRA or Full SFT) and evaluated them, we need to manage them effectively.

This module demonstrates:
1. **Vertex Model Registry**: Finding our tuned models, tagging them with aliases (e.g., `production` or `staging`).
2. **Vertex AI Endpoints**: Deploying the chosen model to a managed endpoint for scalable inference.
3. **Inference**: Calling the deployed endpoint with a test prompt.

In [ ]:
.pip install google-cloud-aiplatform python-dotenv -q

## 1. Setup and Authentication

In [ ]:
import os
from google.cloud import aiplatform
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

PROJECT_ID = os.getenv("PROJECT_ID", "YOUR_PROJECT_ID")
LOCATION = os.getenv("LOCATION", "us-central1")
STAGING_BUCKET = os.getenv("STAGING_BUCKET", "YOUR_GCS_BUCKET_NAME")

if not STAGING_BUCKET.startswith("gs://"):
    STAGING_BUCKET = f"gs://{STAGING_BUCKET}"

aiplatform.init(project=PROJECT_ID, location=LOCATION, staging_bucket=STAGING_BUCKET)

print(f"Project ID: {PROJECT_ID}")
print(f"Location: {LOCATION}")
print(f"Staging Bucket: {STAGING_BUCKET}")

## 2. Vertex Model Registry

Tuning jobs automatically register the resulting model artifacts into the Vertex Model Registry.
We can list our models and apply aliases to them to signify their lifecycle state.

In [ ]:
# To find your exact model name, check the Vertex AI UI or the output of the tuning jobs
# e.g. 'projects/12345/locations/us-central1/models/987654321'
TUNED_MODEL_ID = "YOUR_TUNED_MODEL_ID"

try:
    model = aiplatform.Model(TUNED_MODEL_ID)
    print(f"Found Model: {model.display_name}")
    print(f"Resource Name: {model.resource_name}")
    
    # Add an alias to mark this model as the 'production' evaluation winner
    print("\nAdding 'production' alias..")
    # If 'default' exists, alias modification merges
    # Version aliases help downstream applications decouple from hardcoded model IDs
    # Note: Using try/except as some users might not have permissions to modify aliases
    try: 
        model.version_aliases = ['production', 'latest-cyber-model']
        print("Aliases updated successfully. You can verify this in the Cloud Console.")
    except Exception as modify_e:
        print(f"Could not modify alias (alias update skipped): {modify_e}")
        
except Exception as e:
    print(f"Could not retrieve model: {e}\nEnsure TUNED_MODEL_ID is correct.")

## 3. Vertex AI Endpoints (Deployment)

To use our LoRA or SFT model defensively or offensively in the real world, we deploy it to an endpoint.
Vertex will intelligently pick the required underlying serving image (e.g., vLLM or Triton) automatically for the architecture.

In [ ]:
ENDPOINT_NAME = "cyber-model-endpoint"

# Create an Endpoint resource (think of this as the stable URL/IP address)
try:
    endpoints = aiplatform.Endpoint.list(filter=f'display_name="{ENDPOINT_NAME}"')
    if endpoints:
        endpoint = endpoints[0]
        print(f"Found existing endpoint: {endpoint.name}")
    else:
        print(f"Creating endpoint: {ENDPOINT_NAME}..")
        endpoint = aiplatform.Endpoint.create(display_name=ENDPOINT_NAME)
        print(f"Endpoint created: {endpoint.name}")
        
    # DEPLOY THE MODEL
    # Warning: Deployment can take 15-30 minutes for large LLMs as it spins up GPUs
    print(f"\nDeploying {model.display_name} to endpoint..")
    print("Note: Deployment takes 15-30 minutes based on GPU availability.")
    
    # Uncomment to actually deploy in a live demo environment
    """
    model.deploy(
        endpoint=endpoint,
        machine_type="g2-standard-12", # L4 GPU (ideal for 8B models)
        accelerator_type="NVIDIA_L4",
        accelerator_count=1,
        sync=False # Run asynchronously so the notebook doesn't hang forever
    )
    print("Deployment job submitted asynchronously. Check the Google Cloud Console.")
    """
except NameError:
    print("TUNED_MODEL_ID not defined or model not found in previous step.")
except Exception as e:
    print(f"Endpoint deployment issue: {e}")

## 4. Inference against the Endpoint

Once the deployment finishes turning green in the UI, we can ping it with predictions.

In [ ]:
import json

print("Ensure the endpoint is fully active before running inference.")

test_instance = {
    "prompt": "Act as an expert AI security analyst assistant.\nUser: Explain the attack vector in the log:\n'POST /login.php HTTP/1.1\nHost: example.com\nUser-Agent: sqlmap/1.5.8#dev (http://sqlmap.org)\nContent-Length: 43\n\nusername=admin%27+OR+1%3D1--&password=foo'\nAssistant:",
    "max_tokens": 512,
    "temperature": 0.2,
    "top_p": 0.95,
    "top_k": 40
}

try:
    # Depending on the serving image, the raw predict signature might differ slightly.
    # Often open models expect an instances block.
    # prediction = endpoint.predict(instances=[test_instance])
    # print(prediction.predictions)
    print(f"Mocking request to endpoint {endpoint.name}..")
    print("Prompt:", test_instance['prompt'])
    print("Simulation successful. You can uncomment the predict line above in a live environment.")
except Exception as e:
    print(f"Inference error: {e}")